In [ ]:
# ============================================================
# MAIA LITE - CORPUS V2
# CLEAN NOTEBOOK SETUP
#
# Official continuation:
#     equivalent-140k -> equivalent-216,567
#
# This cell:
#   1. Installs dependencies
#   2. Mounts Google Drive
#   3. Authenticates Hugging Face
#   4. Audits the existing tokenizer
#   5. Copies the official unified Corpus V2 to local SSD
#   6. Verifies byte count and SHA-256
#
# NO TRAINING IS PERFORMED HERE.
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q transformers datasets accelerate tokenizers huggingface_hub


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gc
import time
import shutil
import hashlib
from pathlib import Path

import torch

from google.colab import drive, userdata
from huggingface_hub import login
from transformers import GPT2TokenizerFast


# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive"
)


# ============================================================
# 4. PROJECT
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print()
print(
    f"Project directory:\n"
    f"{PROJECT_DIR}"
)


# ============================================================
# 5. HUGGING FACE AUTHENTICATION
# ============================================================

try:

    hf_token = userdata.get(
        "HF_TOKEN"
    )

    if hf_token:

        login(
            token=hf_token,
            add_to_git_credential=False
        )

        print(
            "Hugging Face authentication: PASS"
        )

    else:

        print(
            "WARNING: HF_TOKEN is empty."
        )

except Exception as error:

    print(
        f"WARNING: Hugging Face authentication "
        f"was not configured: {error}"
    )


# ============================================================
# 6. OFFICIAL PATHS
# ============================================================

TOKENIZER_DIR = (
    PROJECT_DIR
    / "tokenizador"
)

DRIVE_CORPUS = (
    PROJECT_DIR
    / "corpus_v2"
    / "maia_corpus_v2.bin"
)

LOCAL_CORPUS = Path(
    "/content/maia_corpus_v2.bin"
)

BASE_140K = (
    PROJECT_DIR
    / "checkpoints_corpus_v2"
    / "maia_lite_equivalent_140k"
)


# ============================================================
# 7. OFFICIAL CORPUS CONSTANTS
# ============================================================

EXPECTED_TOKENS = 2_899_999_744

EXPECTED_BYTES = 5_799_999_488

EXPECTED_SEQUENCES = 2_832_031

EXPECTED_SHA256 = (
    "57e3a10aac2473dd8af428dba2fd35031410f5e10d77fa3d6933c0cd92503ca2"
)

SEQUENCE_LENGTH = 1024


# ============================================================
# 8. VERIFY CRITICAL DRIVE ARTIFACTS
# ============================================================

print()
print("=" * 88)
print("CRITICAL ARTIFACT AUDIT")
print("=" * 88)


for label, path in [

    ("Tokenizer", TOKENIZER_DIR),

    ("Corpus V2", DRIVE_CORPUS),

    ("Equivalent-140k model", BASE_140K),

]:

    exists = path.exists()

    print(
        f"{label:25s} "
        f"{'PASS' if exists else 'MISSING'}"
    )

    print(
        f"  {path}"
    )

    if not exists:

        raise RuntimeError(
            f"\nRequired artifact missing:\n{path}"
        )


# ============================================================
# 9. TOKENIZER AUDIT
# ============================================================

print()
print("=" * 88)
print("TOKENIZER")
print("=" * 88)


tokenizer = (
    GPT2TokenizerFast.from_pretrained(
        str(TOKENIZER_DIR),
        local_files_only=True
    )
)


print(
    f"Vocabulary:                  "
    f"{len(tokenizer):,}"
)

print(
    f"BOS:                         "
    f"{tokenizer.bos_token_id}"
)

print(
    f"EOS:                         "
    f"{tokenizer.eos_token_id}"
)

print(
    f"PAD:                         "
    f"{tokenizer.pad_token_id}"
)


if len(tokenizer) != 50_257:

    raise RuntimeError(
        "Tokenizer vocabulary mismatch."
    )


if tokenizer.eos_token_id != 2:

    raise RuntimeError(
        "Unexpected EOS token."
    )


if tokenizer.pad_token_id != 1:

    raise RuntimeError(
        "Unexpected PAD token."
    )


print(
    "Tokenizer audit:             PASS"
)


# ============================================================
# 10. DRIVE CORPUS SIZE AUDIT
# ============================================================

print()
print("=" * 88)
print("OFFICIAL DRIVE CORPUS")
print("=" * 88)


drive_bytes = (
    DRIVE_CORPUS.stat().st_size
)


print(
    f"Bytes:                       "
    f"{drive_bytes:,}"
)


if drive_bytes != EXPECTED_BYTES:

    raise RuntimeError(
        "Official Drive corpus has unexpected size."
    )


print(
    "Drive size audit:            PASS"
)


# ============================================================
# 11. COPY CORPUS TO LOCAL SSD
# ============================================================

print()
print("=" * 88)
print("LOCAL SSD CORPUS")
print("=" * 88)


local_valid_size = (

    LOCAL_CORPUS.exists()

    and

    LOCAL_CORPUS.stat().st_size
    == EXPECTED_BYTES
)


if local_valid_size:

    print(
        "Complete-size local corpus already exists."
    )

else:

    if LOCAL_CORPUS.exists():

        print(
            "Removing incomplete local corpus..."
        )

        LOCAL_CORPUS.unlink()


    print(
        "Copying official Corpus V2 "
        "from Drive to local SSD..."
    )


    copy_start = time.time()


    shutil.copyfile(
        DRIVE_CORPUS,
        LOCAL_CORPUS
    )


    copy_elapsed = (
        time.time()
        - copy_start
    )


    print(
        f"Copy time:                   "
        f"{copy_elapsed / 60:.2f} min"
    )


# ============================================================
# 12. LOCAL SIZE AUDIT
# ============================================================

local_bytes = (
    LOCAL_CORPUS.stat().st_size
)

local_tokens = (
    local_bytes // 2
)

local_sequences = (
    local_tokens
    // SEQUENCE_LENGTH
)


print()
print(
    f"Bytes:                       "
    f"{local_bytes:,}"
)

print(
    f"Tokens:                      "
    f"{local_tokens:,}"
)

print(
    f"Sequences:                   "
    f"{local_sequences:,}"
)


if local_bytes != EXPECTED_BYTES:

    raise RuntimeError(
        "Local corpus byte count mismatch."
    )


if local_tokens != EXPECTED_TOKENS:

    raise RuntimeError(
        "Local corpus token count mismatch."
    )


if local_sequences != EXPECTED_SEQUENCES:

    raise RuntimeError(
        "Local corpus sequence count mismatch."
    )


# ============================================================
# 13. SHA-256
# ============================================================

print()
print(
    "Calculating local SHA-256..."
)


digest = hashlib.sha256()

hash_start = time.time()


with open(
    LOCAL_CORPUS,
    "rb"
) as file:

    while True:

        block = file.read(
            64 * 1024 * 1024
        )

        if not block:

            break

        digest.update(
            block
        )


local_sha256 = (
    digest.hexdigest()
)


print(
    f"SHA-256:"
)

print(
    local_sha256
)

print(
    f"Hash time:                   "
    f"{(time.time() - hash_start) / 60:.2f} min"
)


if local_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "\nSHA-256 MISMATCH.\n"
        "DO NOT TRAIN."
    )


# ============================================================
# 14. FINAL
# ============================================================

print()
print("=" * 88)
print("MAIA LITE CLEAN NOTEBOOK SETUP: PASS")
print("=" * 88)

print()

print(
    "Official Corpus V2 is now on local SSD."
)

print(
    "Tokenizer verified."
)

print(
    "Equivalent-140k model verified."
)

print()

print(
    "READY FOR 140K -> 216,567."
)

Mounted at /content/drive

Project directory:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite
Hugging Face authentication: PASS

CRITICAL ARTIFACT AUDIT
Tokenizer                 PASS
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador
Corpus V2                 PASS
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite/corpus_v2/maia_corpus_v2.bin
Equivalent-140k model     PASS
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_corpus_v2/maia_lite_equivalent_140k

TOKENIZER
Vocabulary:                  50,257
BOS:                         0
EOS:                         2
PAD:                         1
Tokenizer audit:             PASS

OFFICIAL DRIVE CORPUS
Bytes:                       5,799,999,488
Drive size audit:            PASS

LOCAL SSD CORPUS
Copying official Corpus V2 from Drive to local SSD...
Copy time:                   1.98 min

Bytes:                       5,799,999,488
Tokens:                      2,899,999,744
Sequences:                   2,832,031

Calculating local SH

In [ ]:
# ============================================================
# MAIA LITE - OFFICIAL CORPUS V2 CONTINUATION
#
# Continuous trajectory:
#
#     equivalent-140,000
#           |
#           v
#     equivalent-216,567
#
# Remaining optimizer steps:
#
#     76,567
#
# Effective batch:
#
#     16 x 2 = 32 sequences
#
# Tokens / optimizer step:
#
#     32 x 1024 = 32,768
#
# This run DOES NOT intentionally stop at intermediate
# milestones.
#
# ============================================================


# ============================================================
# 1. ENVIRONMENT
# ============================================================

import os

os.environ[
    "PYTORCH_CUDA_ALLOC_CONF"
] = "expandable_segments:True"


# ============================================================
# 2. IMPORTS
# ============================================================

import gc
import re
import time
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import Dataset

from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 42

set_seed(
    SEED
)


# ============================================================
# 4. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
)

TOKENIZER_DIR = (
    PROJECT_DIR
    / "tokenizador"
)

LOCAL_CORPUS = Path(
    "/content/maia_corpus_v2.bin"
)

BASE_140K = (
    PROJECT_DIR
    / "checkpoints_corpus_v2"
    / "maia_lite_equivalent_140k"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "checkpoints_corpus_v2_140k_to_216567"
)

FINAL_DIR = (
    OUTPUT_DIR
    / "maia_lite_equivalent_216567"
)


# ============================================================
# 5. MODEL / CORPUS CONSTANTS
# ============================================================

SEQUENCE_LENGTH = 1024

EXPECTED_CORPUS_BYTES = (
    5_799_999_488
)

EXPECTED_SEQUENCES = (
    2_832_031
)


# ============================================================
# 6. TRAINING CONSTANTS
# ============================================================

START_EQUIVALENT_STEP = (
    140_000
)

FINAL_EQUIVALENT_STEP = (
    216_567
)

TOTAL_STEPS = (
    FINAL_EQUIVALENT_STEP
    - START_EQUIVALENT_STEP
)


MICRO_BATCH = 16

GRAD_ACCUM = 2

EFFECTIVE_BATCH = (
    MICRO_BATCH
    * GRAD_ACCUM
)

TOKENS_PER_STEP = (
    EFFECTIVE_BATCH
    * SEQUENCE_LENGTH
)


LEARNING_RATE = 1e-5

WARMUP_STEPS = 250

WEIGHT_DECAY = 0.1

MAX_GRAD_NORM = 1.0


SAVE_STEPS = 1000

LOGGING_STEPS = 100


# ============================================================
# 7. STATIC CHECKS
# ============================================================

if TOTAL_STEPS != 76_567:

    raise RuntimeError(
        "Trajectory step calculation failed."
    )


if EFFECTIVE_BATCH != 32:

    raise RuntimeError(
        "Effective batch must equal 32."
    )


if TOKENS_PER_STEP != 32_768:

    raise RuntimeError(
        "Tokens per optimizer step must equal 32,768."
    )


if not LOCAL_CORPUS.exists():

    raise RuntimeError(
        "Local Corpus V2 is missing. "
        "Run Cell 1 first."
    )


if (
    LOCAL_CORPUS.stat().st_size
    != EXPECTED_CORPUS_BYTES
):

    raise RuntimeError(
        "Local Corpus V2 size mismatch."
    )


# ============================================================
# 8. GPU
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU not available."
    )


gc.collect()

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


gpu_name = (
    torch.cuda.get_device_name(0)
)

gpu_memory = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / 1024**3
)


print("=" * 92)
print("MAIA LITE - OFFICIAL 140K -> 216,567 CONTINUATION")
print("=" * 92)

print(
    f"GPU:                         "
    f"{gpu_name}"
)

print(
    f"GPU memory:                  "
    f"{gpu_memory:.1f} GB"
)

print(
    f"PyTorch:                     "
    f"{torch.__version__}"
)

print(
    f"CUDA:                        "
    f"{torch.version.cuda}"
)


# ============================================================
# 9. DATASET
# ============================================================

class MaiaCorpusDataset(Dataset):

    def __init__(
        self,
        path,
        sequence_length
    ):

        self.sequence_length = (
            sequence_length
        )

        self.data = np.memmap(
            path,
            dtype=np.uint16,
            mode="r"
        )

        self.sequence_count = (
            len(self.data)
            // self.sequence_length
        )


    def __len__(
        self
    ):

        return (
            self.sequence_count
        )


    def __getitem__(
        self,
        index
    ):

        start = (
            index
            * self.sequence_length
        )

        end = (
            start
            + self.sequence_length
        )


        input_ids = np.array(
            self.data[start:end],
            dtype=np.int64,
            copy=True
        )


        return {
            "input_ids": input_ids
        }


train_dataset = MaiaCorpusDataset(
    LOCAL_CORPUS,
    SEQUENCE_LENGTH
)


if len(train_dataset) != EXPECTED_SEQUENCES:

    raise RuntimeError(
        "Dataset sequence count mismatch."
    )


# ============================================================
# 10. TOKENIZER
# ============================================================

tokenizer = (
    GPT2TokenizerFast.from_pretrained(
        str(TOKENIZER_DIR),
        local_files_only=True
    )
)


if len(tokenizer) != 50_257:

    raise RuntimeError(
        "Tokenizer vocabulary mismatch."
    )


if tokenizer.eos_token_id != 2:

    raise RuntimeError(
        "EOS mismatch."
    )


if tokenizer.pad_token_id != 1:

    raise RuntimeError(
        "PAD mismatch."
    )


data_collator = (
    DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)


# ============================================================
# 11. CHECKPOINT DISCOVERY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def checkpoint_number(
    path
):

    match = re.fullmatch(
        r"checkpoint-(\d+)",
        path.name
    )

    if match is None:

        return -1

    return int(
        match.group(1)
    )


checkpoints = [

    path

    for path in OUTPUT_DIR.glob(
        "checkpoint-*"
    )

    if (
        path.is_dir()
        and
        checkpoint_number(path) >= 0
    )
]


checkpoints.sort(
    key=checkpoint_number
)


latest_checkpoint = (

    checkpoints[-1]

    if checkpoints

    else None
)


# ============================================================
# 12. SELECT START MODE
# ============================================================

print()
print("=" * 92)
print("START MODE")
print("=" * 92)


if latest_checkpoint is None:

    if not BASE_140K.exists():

        raise RuntimeError(
            f"Official 140k model missing:\n"
            f"{BASE_140K}"
        )


    print(
        "Mode:                        NEW TRAJECTORY"
    )

    print(
        "Starting model:              equivalent-140k"
    )

    print(
        "Optimizer:                   NEW"
    )

    print(
        "Scheduler:                   NEW"
    )

    print(
        "Trainer state:               NEW"
    )


    model_path = (
        BASE_140K
    )

    resume_from_checkpoint = None


else:

    resume_step = (
        checkpoint_number(
            latest_checkpoint
        )
    )


    if resume_step > TOTAL_STEPS:

        raise RuntimeError(
            "Checkpoint is beyond the "
            "configured trajectory."
        )


    print(
        "Mode:                        FULL RESUME"
    )

    print(
        f"Checkpoint:                  "
        f"{latest_checkpoint}"
    )

    print(
        f"Trajectory step:             "
        f"{resume_step:,}"
    )

    print(
        f"Equivalent Maia step:        "
        f"{START_EQUIVALENT_STEP + resume_step:,}"
    )

    print(
        "Optimizer:                   RESTORE"
    )

    print(
        "Scheduler:                   RESTORE"
    )

    print(
        "Trainer state:               RESTORE"
    )


    model_path = (
        latest_checkpoint
    )

    resume_from_checkpoint = str(
        latest_checkpoint
    )


# ============================================================
# 13. MODEL
# ============================================================

print()
print("=" * 92)
print("MODEL")
print("=" * 92)


model = (
    GPT2LMHeadModel.from_pretrained(
        str(model_path),
        torch_dtype=torch.float32
    )
)


model.gradient_checkpointing_enable()

model.config.use_cache = False


parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)


print(
    f"Parameters:                  "
    f"{parameter_count:,}"
)

print(
    "Gradient checkpointing:      ENABLED"
)

print(
    "KV cache:                    DISABLED"
)


if parameter_count != 354_823_168:

    raise RuntimeError(
        "Unexpected model parameter count."
    )


# ============================================================
# 14. TRAINING ARGUMENTS
#
# max_steps is ALWAYS 76,567.
#
# Therefore, a resumed run reconstructs the same scheduler
# horizon before loading scheduler state.
# ============================================================

training_args = TrainingArguments(

    output_dir=str(
        OUTPUT_DIR
    ),

    max_steps=TOTAL_STEPS,

    per_device_train_batch_size=(
        MICRO_BATCH
    ),

    gradient_accumulation_steps=(
        GRAD_ACCUM
    ),

    learning_rate=(
        LEARNING_RATE
    ),

    warmup_steps=(
        WARMUP_STEPS
    ),

    lr_scheduler_type="cosine",

    weight_decay=(
        WEIGHT_DECAY
    ),

    max_grad_norm=(
        MAX_GRAD_NORM
    ),

    bf16=True,

    fp16=False,

    tf32=True,

    seed=SEED,

    data_seed=SEED,

    save_strategy="steps",

    save_steps=(
        SAVE_STEPS
    ),

    save_total_limit=None,

    logging_strategy="steps",

    logging_steps=(
        LOGGING_STEPS
    ),

    logging_first_step=True,

    eval_strategy="no",

    dataloader_num_workers=0,

    dataloader_pin_memory=True,

    report_to="none",

    remove_unused_columns=False,
)


# ============================================================
# 15. TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    data_collator=data_collator
)


# ============================================================
# 16. PRE-FLIGHT
# ============================================================

already_done = (

    checkpoint_number(
        latest_checkpoint
    )

    if latest_checkpoint

    else 0
)


remaining_steps = (
    TOTAL_STEPS
    - already_done
)


print()
print("=" * 92)
print("OFFICIAL PRE-FLIGHT")
print("=" * 92)

print(
    f"Starting equivalent step:    "
    f"{START_EQUIVALENT_STEP:,}"
)

print(
    f"Final equivalent step:       "
    f"{FINAL_EQUIVALENT_STEP:,}"
)

print(
    f"Trajectory length:           "
    f"{TOTAL_STEPS:,}"
)

print(
    f"Already completed:           "
    f"{already_done:,}"
)

print(
    f"Remaining:                   "
    f"{remaining_steps:,}"
)

print()

print(
    f"Micro batch:                 "
    f"{MICRO_BATCH}"
)

print(
    f"Gradient accumulation:       "
    f"{GRAD_ACCUM}"
)

print(
    f"Effective batch:             "
    f"{EFFECTIVE_BATCH}"
)

print(
    f"Tokens / optimizer step:     "
    f"{TOKENS_PER_STEP:,}"
)

print()

print(
    f"Maximum LR:                  "
    f"{LEARNING_RATE:.10f}"
)

print(
    f"Warmup:                      "
    f"{WARMUP_STEPS:,}"
)

print(
    "Scheduler:                   cosine"
)

print(
    f"Scheduler horizon:           "
    f"{TOTAL_STEPS:,}"
)

print()

print(
    f"Total trajectory tokens:     "
    f"{TOTAL_STEPS * TOKENS_PER_STEP:,}"
)

print(
    f"Remaining tokens this run:   "
    f"{remaining_steps * TOKENS_PER_STEP:,}"
)

print()

print(
    f"Dataset sequences:           "
    f"{len(train_dataset):,}"
)

print(
    f"Corpus bytes:                "
    f"{LOCAL_CORPUS.stat().st_size:,}"
)

print()

print(
    "Official 140k validation:"
)

print(
    "  loss = 3.60799408"
)

print(
    "  PPL  = 36.891976"
)

print()

print(
    "Checkpoints every:           "
    f"{SAVE_STEPS:,} optimizer steps"
)

print()

print(
    "NO scheduled intermediate stop."
)


# ============================================================
# 17. TRAIN
# ============================================================

print()
print("=" * 92)
print("STARTING CONTINUOUS PRETRAINING")
print("=" * 92)


torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


training_start = (
    time.time()
)


train_result = trainer.train(
    resume_from_checkpoint=(
        resume_from_checkpoint
    )
)


torch.cuda.synchronize()


training_elapsed = (
    time.time()
    - training_start
)


# ============================================================
# 18. VERIFY FINAL STEP
# ============================================================

final_global_step = (
    trainer.state.global_step
)


if final_global_step != TOTAL_STEPS:

    raise RuntimeError(
        f"\nTraining stopped at trajectory step "
        f"{final_global_step:,}, not {TOTAL_STEPS:,}.\n\n"
        "The latest full checkpoint remains available "
        "for resume."
    )


equivalent_step = (
    START_EQUIVALENT_STEP
    + final_global_step
)


if equivalent_step != FINAL_EQUIVALENT_STEP:

    raise RuntimeError(
        "Equivalent-step accounting error."
    )


# ============================================================
# 19. SAVE FINAL MODEL
# ============================================================

print()
print(
    "Saving final equivalent-216567 model..."
)


trainer.save_model(
    str(FINAL_DIR)
)


tokenizer.save_pretrained(
    str(FINAL_DIR)
)


# ============================================================
# 20. FINAL REPORT
# ============================================================

peak_allocated = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

peak_reserved = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)


print()
print("=" * 92)
print("MAIA LITE CORPUS V2 PRETRAINING COMPLETE")
print("=" * 92)

print(
    f"Trajectory steps:            "
    f"{final_global_step:,}"
)

print(
    f"Equivalent Maia step:        "
    f"{equivalent_step:,}"
)

print(
    f"Corpus V2 tokens processed:  "
    f"{final_global_step * TOKENS_PER_STEP:,}"
)

print(
    f"Runtime:                     "
    f"{training_elapsed / 3600:.2f} h"
)

print(
    f"Peak allocated VRAM:         "
    f"{peak_allocated:.2f} GB"
)

print(
    f"Peak reserved VRAM:          "
    f"{peak_reserved:.2f} GB"
)

print()

print(
    "Final model:"
)

print(
    FINAL_DIR
)

print()

print(
    "NEXT:"
)

print(
    "Run the frozen Validation Corpus V2 "
    "evaluation on equivalent-216567."
)

MAIA LITE - OFFICIAL 140K -> 216,567 CONTINUATION
GPU:                         NVIDIA A100-SXM4-80GB
GPU memory:                  79.3 GB
PyTorch:                     2.11.0+cu128
CUDA:                        12.8

START MODE
Mode:                        NEW TRAJECTORY
Starting model:              equivalent-140k
Optimizer:                   NEW
Scheduler:                   NEW
Trainer state:               NEW

MODEL


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Parameters:                  354,823,168
Gradient checkpointing:      ENABLED
KV cache:                    DISABLED

OFFICIAL PRE-FLIGHT
Starting equivalent step:    140,000
Final equivalent step:       216,567
Trajectory length:           76,567
Already completed:           0
Remaining:                   76,567

Micro batch:                 16
Gradient accumulation:       2
Effective batch:             32
Tokens / optimizer step:     32,768

Maximum LR:                  0.0000100000
Warmup:                      250
Scheduler:                   cosine
Scheduler horizon:           76,567

Total trajectory tokens:     2,508,947,456
Remaining tokens this run:   2,508,947,456

Dataset sequences:           2,832,031
Corpus bytes:                5,799,999,488

Official 140k validation:
  loss = 3.60799408
  PPL  = 36.891976

Checkpoints every:           1,000 optimizer steps

NO scheduled intermediate stop.

STARTING CONTINUOUS PRETRAINING


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.668790
100,3.687766
200,3.667158
300,3.675061
400,3.670409
500,3.663233
600,3.668834
700,3.671364
800,3.659619
900,3.662607


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]